<a href="https://colab.research.google.com/github/MeLBiN122/AI-ML/blob/main/Movie_Review(Day_7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/content/IMDB Dataset.csv", encoding = "latin1", on_bad_lines='skip', engine='python')

In [ ]:
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [ ]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [ ]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


In [ ]:
negation_words = [
    "not good", "not bad", "not great", "don't like "
    "didn't like", "never liked", "wasn't good",
    "isn't good", "no good"
]

In [ ]:
import re

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zA-Z\s']"," ", text)

  for phrase in negation_words:
    text = text.replace(phrase, phrase.replace(" ", "_"))

  return text

In [ ]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['review'], df['sentiment'], test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words = vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen = max_len, padding = 'post')
X_test_pad = pad_sequences(X_test_seq, maxlen = max_len, padding = 'post')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size, 128, input_length = max_len),

    LSTM(128, dropout = 0.3, recurrent_dropout = 0.3),

    Dense(64, activation = 'relu'),
    Dropout(0.5),

    Dense(1, activation = 'sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_pad, y_train,
    epochs = 5,
    batch_size = 64,
    validation_split =  0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 443s 876ms/step - accuracy: 0.5512 - loss: 0.6677 - val_accuracy: 0.5931 - val_loss: 0.6175
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 438s 868ms/step - accuracy: 0.6022 - loss: 0.6066 - val_accuracy: 0.5995 - val_loss: 0.6192
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 483s 967ms/step - accuracy: 0.7417 - loss: 0.5279 - val_accuracy: 0.8151 - val_loss: 0.4808
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 495s 952ms/step - accuracy: 0.7036 - loss: 0.5385 - val_accuracy: 0.6774 - val_loss: 0.5908
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 435s 871ms/step - accuracy: 0.8288 - loss: 0.3967 - val_accuracy: 0.8714 - val_loss: 0.3388


In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 109ms/step - accuracy: 0.8688 - loss: 0.3447
Test Accuracy: 0.8687999844551086


In [ ]:
def predict_sentiment(review):
  review = clean_text(review)
  seq = tokenizer.texts_to_sequences([review])
  padded = pad_sequences(seq, maxlen = max_len, padding = 'post')
  prediction = model.predict(padded)[0][0]
  print("\nReview: ",review)

  if prediction >= 0.5:
    print("Sentiment: Positive 😍")
  else:
    print("Sentiment: Negative 🙂")

In [ ]:
predict_sentiment("This movie was absolutely anazing and I loved it!!👏")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step

Review:  this movie was absolutely anazing and i loved it   
Sentiment: Positive 😍
